In [45]:
import aspose.words_foss as aw 

docs_path = './docs/'
filename = "uk-rf.doc"

doc = aw.Document(docs_path+filename)
text = doc.get_text()

In [89]:
text[:1000]

'Уголовный кодекс Российской Федерации от 13 июня 1996 г. N 63-ФЗ\nПринят Государственной Думой 24 мая 1996 года\nОдобрен Советом Федерации 5 июня 1996 года\nОбщая часть\nРаздел I. Уголовный закон\nГлава 1. Задачи и принципы Уголовного кодекса Российской Федерации\nСтатья 1. Уголовное законодательство Российской Федерации\n1. Уголовное законодательство Российской Федерации состоит из настоящего Кодекса. Новые законы, предусматривающие уголовную ответственность, подлежат включению в настоящий Кодекс.\n2. Настоящий Кодекс основывается на Конституции Российской Федерации и общепризнанных принципах и нормах международного права.\nСтатья 2. Задачи Уголовного кодекса Российской Федерации\n1. Задачами настоящего Кодекса являются: охрана прав и свобод человека и гражданина, собственности, общественного порядка и общественной безопасности, окружающей среды, конституционного строя Российской Федерации от преступных посягательств, обеспечение мира и безопасности человечества, а также предупрежден

In [46]:
import numpy as np

def split_into_chunks(text: str, chunk_size: int,  overlap: int=0) -> list[str]:
    # для каждого чанка выведем начальный и конечный индексы, чтобы 
    if overlap >= chunk_size:
        raise ValueError("chunk_size должен быть больше overlap!")

    if overlap < 0:
        raise ValueError("overlap не должен быть меньше нуля!")

    if chunk_size < 0:
        raise ValueError("chunk_size не может быть меньше нуля!")

    if not text:
        raise ValueError("Текст пустой!")

    start_indexes = np.arange(0, len(text), chunk_size-overlap)
    end_indexes = start_indexes + chunk_size

    chunks = [
        text[start:end] for start, end in zip(start_indexes, end_indexes)
    ]

    return chunks

In [47]:
import torch
import torch.nn.functional as F
import polars as pl
import hashlib
from transformers import AutoTokenizer, AutoModel

pl.Config.set_fmt_str_lengths(1000)
torch.set_grad_enabled(False)
device = "cuda" if torch.cuda.is_available() else "cpu"

cache_dir = './models/'
embeding_model_name = "google/embeddinggemma-2"

vdb_path = './database/'
vdb_filename = 'test.parquet'

tokenizer = AutoTokenizer.from_pretrained(embeding_model_name, cache_dir=cache_dir)
model = AutoModel.from_pretrained(embeding_model_name, cache_dir=cache_dir).to(device).eval()

def get_ids(chunks, tokenizer):
    tokens = [tokenizer.tokenize(chunk) for chunk in chunks]

    ids = [tokenizer.convert_tokens_to_ids(token) for token in tokens]

    prepared_ids = tokenizer.pad(
        {"input_ids": ids},
        padding=True,
        return_attention_mask=True,
        return_tensors="pt"
    )

    return prepared_ids

def get_embeddings(batch, model):
    with torch.inference_mode():
        output = model(**batch)

    token_embeddings = output.last_hidden_state
    token_embeddings_prepared = (
        batch['attention_mask']
        .to(device)
        .to(torch.float64)
        .unsqueeze(-1)
    ) * (
        token_embeddings
    )

    return token_embeddings_prepared.mean(dim=1).cpu()

overlap = 60
chunk_size = 800

chunks = split_into_chunks(text, chunk_size, overlap)

ids = tokenizer(
    chunks,
    return_tensors="pt",
    padding=True
)

batch_size = 128
tensor_len = ids['input_ids'].shape[0]

embeddings = []

for i in np.arange(0, tensor_len, batch_size):
    batch = {
        key: value[i:i+batch_size].to(device)
        for key, value in ids.items()
    }

    with torch.inference_mode():
        output = model(**batch).last_hidden_state

    embeddings.append(output.to("cpu"))

embeddings = torch.concat(embeddings, dim=0)
embeddings = (embeddings * ids['attention_mask'].unsqueeze(-1)).sum(1) / ids['attention_mask'].sum(1, keepdim=True)
embeddings = F.normalize(embeddings, dim=1)

del output; del model; del tokenizer

table = pl.DataFrame({
    "id": [hashlib.sha256(chunk.encode('utf-8')).hexdigest() for chunk in chunks],
    "text": chunks,
    "embedding_vector": embeddings.to(torch.float64).numpy(),
    "doc": filename
})

table.write_parquet(
    vdb_path+vdb_filename
)

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

In [61]:
# search in base
query1 = "Мне нужна узнать: что такое кража?"
query2 = "В какой статье есть наказание за убийство?"
query3 = "Какой кодекс в арбитражном суде?"

def search(query, model, tokenizer, base_file):
    # tokenizer

    # embedding_vector = get_embeddings()
    pass

In [62]:
tokenizer = AutoTokenizer.from_pretrained(embeding_model_name, cache_dir=cache_dir)
model = AutoModel.from_pretrained(embeding_model_name, cache_dir=cache_dir).to(device).eval()

tokens = tokenizer(
    query1, 
    attention_mask=True,
    return_tensors="pt"
).to(device)

output = F.normalize(model(**tokens).last_hidden_state, dim=1)
output = output.mean(dim=1).to(torch.float64).cpu()

Loading weights:   0%|          | 0/1376 [00:00<?, ?it/s]

In [63]:
def cosine_similarity(x, y):
    return x @ y

db = pl.read_parquet(vdb_path+vdb_filename)
vector_db = torch.tensor(db['embedding_vector'].to_list(), dtype=torch.float64)
top_k_scores, top_k_indices = torch.topk(vector_db @ output.squeeze(0), k=40)

result = (
    db[top_k_indices.cpu().numpy()]
)

In [64]:
# rerank
from sentence_transformers import CrossEncoder

reranker_name = "Qwen/Qwen3-Reranker-0.6B"
reranking_model = CrossEncoder(reranker_name, cache_folder=cache_dir, device=device)

pairs = [(query1, doc) for doc in result['text']]
scores = reranking_model.predict(pairs)
print(scores)

Loading weights:   0%|          | 0/310 [00:00<?, ?it/s]

[-4.625   -5.3125  -4.375    4.125    3.6875   3.1875  -4.875   -4.625
  4.      -5.59375 -5.53125 -4.375   -4.1875  -4.625   -5.375   -5.5
 -4.6875  -4.78125 -5.3125  -6.25    -5.625   -6.75    -4.875   -5.1875
 -4.8125  -5.0625  -6.15625 -5.96875 -5.09375 -5.25    -4.5625  -4.5
 -6.59375 -4.9375  -0.25    -5.96875 -6.25    -5.75    -4.5625  -4.125  ]


In [65]:
print(scores.argmax(), scores[scores.argmax()])

3 4.125


In [66]:
db[int(scores.argmax())]

id,text,embedding_vector,doc
str,str,"array[f64, 768]",str
"""0694e31dc2ee10de6bc93aaea08a7009a50e57341deac03e52415d8213ba3ed4""","""тья 6. Принцип справедливости 1. Наказание и иные меры уголовно-правового характера, применяемые к лицу, совершившему преступление, должны быть справедливыми, то есть соответствовать характеру и степени общественной опасности преступления, обстоятельствам его совершения и личности виновного. 2. Никто не может нести уголовную ответственность дважды за одно и то же преступление. Статья 7. Принцип гуманизма 1. Уголовное законодательство Российской Федерации обеспечивает безопасность человека. 2. Наказание и иные меры уголовно-правового характера, применяемые к лицу, совершившему преступление, не могут иметь своей целью причинение физических страданий или унижение человеческого достоинства. Статья 8. Основание уголовной ответственности Основанием уголовной ответственности является совершение д""","[-0.007385, -0.014343, … 0.004974]","""uk-rf.doc"""


In [ ]:
# Generate answer
